In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ================== CONFIG ==================
PER_DEFECT_CSV = "anomalib_results_per_defect.csv"
FULL_CSV = "anomalib_results_by_dataset.csv"
PER_DEFECT_PDF = "per_defect_f1.pdf"
ALL_DEFECTS_PDF = "all_defects_f1.pdf"
DECIMALS = 2

PREFIX='md' # or 'md' or 'cq'



# ================== STYLE ==================
FONT = {
    "title": 16,
    "axis": 14,
    "ticks": 14,
    "legend": 14,
    "annotation": 11,
}

sns.set(style="whitegrid")

# ================== LOAD DATA ==================
df_per_defect = pd.read_csv(PER_DEFECT_CSV)
df_full = pd.read_csv(FULL_CSV)

for df in (df_per_defect, df_full):
    df["image_F1Score"] = pd.to_numeric(df["image_F1Score"], errors="coerce")

# ================== DEFECT MAPPING ==================

if PREFIX == 'cq':
    defect_map = {
        'cq_public_damaged':"Damaged", 
        'cq_public_dirty':"Dirty", 
        'cq_public_oxidized':"Oxidation",
        'cq_public_all_unfit':"All Defects"
        }
    defect_order = ["Damaged", "Dirty", "Oxidation", "All Defects"]
elif PREFIX == 'md':
    defect_map = {
        'md_public_7_partial':'7Partial',
        'md_public_building_partial':'BuildingPartial', 
        'md_public_different_pressures':"DifferentPressures",
        'md_public_star_partial':'StarPartial', 
        'md_public_all_defectives':'All Defects'
    }
    defect_order = ["7Partial", "BuildingPartial", "DifferentPressures", "StarPartial",  "All Defects"]

df_per_defect["Defect"] = df_per_defect["experiment_name"].map(defect_map)
df_individual = df_per_defect[df_per_defect["Defect"].notna()].copy()

models = sorted(df_full["model_name"].unique())

# ================== PLOTTING CONFIG ==================
PLOT_TYPE = "bar"  # "box" or "bar"
CI = 95            # confidence interval for bar plots

# ================== HELPERS ==================
def annotate_bar_means(ax, decimals=2, offset=0.01):
    """Annotate seaborn barplots with mean values."""
    for patch in ax.patches:
        height = patch.get_height()
        if height is None or np.isnan(height) or height <= 0:
            continue
        x = patch.get_x() + patch.get_width() / 2
        y = height + offset
        if x < ax.get_xlim()[0] or x > ax.get_xlim()[1]:
            continue
        ax.text(
            x, y, f"{height:.{decimals}f}",
            ha="center", va="bottom",
            fontsize=FONT["annotation"], color="black", clip_on=True
        )

def apply_fonts(ax):
    """Apply consistent font sizes."""
    ax.set_xlabel(ax.get_xlabel(), fontsize=FONT["axis"])
    ax.set_ylabel(ax.get_ylabel(), fontsize=FONT["axis"])
    ax.set_title(ax.get_title(), fontsize=FONT["title"])
    ax.tick_params(axis="both", labelsize=FONT["ticks"])
    leg = ax.get_legend()
    if leg is not None:
        for text in leg.get_texts():
            text.set_fontsize(FONT["legend"])
        leg.get_title().set_fontsize(FONT["legend"])

# ================== PLOTTING FUNCTIONS ==================
def plot_per_defect():
    if not df_individual["image_F1Score"].notna().any():
        return

    fig, ax = plt.subplots(figsize=(12, 7))

    if PLOT_TYPE == "box":
        sns.boxplot(
            data=df_individual,
            x="Defect",
            y="image_F1Score",
            hue="model_name",
            order=defect_order[:-1],
            hue_order=models,
            palette="Set2",
            linewidth=1.5,
            ax=ax
        )
    elif PLOT_TYPE == "bar":
        sns.barplot(
            data=df_individual,
            x="Defect",
            y="image_F1Score",
            hue="model_name",
            order=defect_order[:-1],
            hue_order=models,
            palette="Set2",
            errorbar=("ci", CI),
            capsize=0.12,
            errwidth=1.2,
            ax=ax
        )
        annotate_bar_means(ax, decimals=DECIMALS)
    else:
        raise ValueError("PLOT_TYPE must be 'box' or 'bar'")

    ax.set_title(
        f"Model Performance on Individual Defect Types\n"
        f"(Image-level F1 Score, {'5 runs' if PLOT_TYPE=='box' else f'{CI}% CI'})"
    )
    ax.set_ylabel("Image F1 Score")
    ax.set_xlabel("Defect Type")
    ax.set_ylim(0.0, 1.02)

    apply_fonts(ax)
    ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(PER_DEFECT_PDF, bbox_inches="tight")
    plt.show()

def plot_all_defects():
    if not df_full["image_F1Score"].notna().any():
        return

    fig, ax = plt.subplots(figsize=(10, 7))

    if PLOT_TYPE == "box":
        sns.boxplot(
            data=df_full,
            x="model_name",
            y="image_F1Score",
            order=models,
            palette="Set2",
            linewidth=1.5,
            ax=ax
        )
    elif PLOT_TYPE == "bar":
        sns.barplot(
            data=df_full,
            x="model_name",
            y="image_F1Score",
            order=models,
            palette="Set2",
            errorbar=("ci", CI),
            capsize=0.12,
            errwidth=1.2,
            ax=ax
        )
        annotate_bar_means(ax, decimals=DECIMALS)
    else:
        raise ValueError("PLOT_TYPE must be 'box' or 'bar'")

    ax.set_title(
        f"Model Performance on All Defects Combined\n"
        f"(Image-level F1 Score, {'full dataset' if PLOT_TYPE=='box' else f'{CI}% CI'})"
    )
    ax.set_ylabel("Image F1 Score")
    ax.set_xlabel("Model")
    ax.set_ylim(0.0, 1.02)
    ax.tick_params(axis="x", rotation=45)

    apply_fonts(ax)
    plt.tight_layout()
    plt.savefig(ALL_DEFECTS_PDF, bbox_inches="tight")
    plt.show()

# ================== RUN PLOTS ==================
plot_per_defect()
plot_all_defects()


In [ ]:
import pandas as pd
import numpy as np

# ================== CONFIG ==================
PER_DEFECT_CSV = "anomalib_results_per_defect.csv"
DECIMALS = 4

# ================== LOAD DATA ==================
df_per_defect = pd.read_csv(PER_DEFECT_CSV)

for df in (df_per_defect, df_full):
    df["image_F1Score"] = pd.to_numeric(df["image_F1Score"], errors="coerce")

# ================== DEFECT MAPPING ==================

if PREFIX == 'cq':
    defect_map = {
        'cq_public_damaged':"Damaged", 
        'cq_public_dirty':"Dirty", 
        'cq_public_oxidized':"Oxidation",
        'cq_public_all_unfit':"All Defects"
        }
    defect_order = ["Damaged", "Dirty", "Oxidation", "All Defects"]
elif PREFIX == 'md':
    defect_map = {
        'md_public_7_partial':'7Partial',
        'md_public_building_partial':'BuildingPartial', 
        'md_public_different_pressures':"DifferentPressures",
        'md_public_star_partial':'StarPartial', 
        'md_public_all_defectives':'All Defects'
    }
    defect_order = ["7Partial", "BuildingPartial", "DifferentPressures", "StarPartial",  "All Defects"]

df_per_defect["Defect"] = df_per_defect["experiment_name"].map(defect_map)
df_individual = df_per_defect[df_per_defect["Defect"].notna()].copy()

models = sorted(df_full["model_name"].unique())
n_models = len(models)

# ================== SUMMARY ==================
def summarize_f1(df, groupby):
    return (
        df.groupby(groupby, observed=True)
        .agg(
            mean=("image_F1Score", "mean"),
            std=("image_F1Score", "std"),
        )
        .round(DECIMALS)
        .reset_index()
    )

summary_per = summarize_f1(df_individual, ["Defect", "model_name"])
#summary_full = summarize_f1(df_full, ["model_name"])
#summary_full["Defect"] = "All Defects"

summary = summary_per#pd.concat([summary_per, summary_full], ignore_index=True)

# ================== FORMAT ==================
fmt = f".{DECIMALS}f"

def format_f1(mean, std):
    if pd.isna(mean):
        return "-"
    if pd.isna(std) or std == 0:
        return f"{mean:{fmt}}"
    return f"{mean:{fmt}} $\\pm$ {std:{fmt}}"

summary["F1_str"] = summary.apply(
    lambda r: format_f1(r["mean"], r["std"]), axis=1
)

# ================== WIDE TABLE ==================
table_df = (
    summary
    .set_index(["Defect", "model_name"])["F1_str"]
    .unstack("model_name")
    .reset_index()
)

table_df["Defect"] = pd.Categorical(
    table_df["Defect"], categories=defect_order, ordered=True
)
table_df = table_df.sort_values("Defect").reset_index(drop=True)

# ================== BOLD BEST F1 ==================
def extract_mean(val):
    try:
        return float(val.split()[0])
    except Exception:
        return np.nan

for defect in table_df["Defect"].unique():
    mask = table_df["Defect"] == defect
    row = table_df.loc[mask]

    vals = {
        m: extract_mean(row[m].iloc[0])
        for m in models
        if m in table_df.columns
    }
    vals = {k: v for k, v in vals.items() if not np.isnan(v)}
    if not vals:
        continue

    best = max(vals.values())
    for m, v in vals.items():
        if v == best:
            table_df.loc[mask, m] = rf"\textbf{{{table_df.loc[mask, m].iloc[0]}}}"

# ================== LATEX ==================
column_format = "l" + "c" * n_models

latex = table_df.to_latex(
    index=False,
    escape=False,
    column_format=column_format,
    caption="Anomaly Detection Performance on Coin Defects (Image-level F1 Score)",
    label="tab:coin_f1_results",
    na_rep="-",
)

print(latex)

print(rf"""
\begin{{tablenotes}}
\item Results are mean $\pm$ standard deviation over 10 runs (rounded to {DECIMALS} decimals).
\item Best F1 score per defect type is shown in \textbf{{bold}}.
\item ``All Defects'' uses the full test set.
\item Threshold for F1 Score was adaptively tuned on validation data.
\end{{tablenotes}}
\end{{table*}}
""")
